# 🧹 Notebook 02 : Nettoyage, Structuration et Couche Silver (Mobilité & Géomarketing - Lyon)

### 🎯 Objectifs de ce carnet :
Ce carnet prend la suite du pipeline d'ingestion (Couche Bronze) pour transformer les données brutes en **données exploitables et qualifiées (Couche Silver)**. 

L'ambition est de préparer le terrain pour le Machine Learning et l'analyse spatiale en séparant les données selon leur rôle fonctionnel.

**Les 4 tables cibles finales (Les Grands Sujets Silver) :**
1. `silver_offre_mobilite` : Tous les équipements physiques (vélos, voitures, TC, parkings, IRVE).
2. `silver_poi_flux` : Les générateurs de flux (santé, commerces, entreprises) pour enrichir le modèle.
3. `silver_demande_carreaux` : La démographie et la motorisation (Filosofi) pour cibler les besoins.
4. `silver_contexte_spatial` : La ZFE (et plus tard la voirie) pour les contraintes réglementaires et le routage.

---

### 🏛️ Le Cadre Méthodologique (Les 8 Règles d'Or)

Pour maintenir un code propre, reproductible et auditable, ce carnet respecte strictement les principes suivants :

1. **Partir de la table finale :** Définir d'abord les schémas cibles pour savoir exactement comment formater chaque source.
2. **Profiler avant de nettoyer (EDA) :** Auditer chaque source pour fixer la criticité avant de transformer.
3. **Quelques recettes, pas douze codes :** Grouper les sources par familles et appliquer une recette générique.
4. **Ne rien supprimer en silence :** Isoler systématiquement les lignes rejetées dans une **table de quarantaine** avec la raison précise du rejet.
5. **Des contrôles qui s'arrêtent bruyamment (*Fail loud*) :** Implanter des assertions bloquantes pour stopper le carnet en cas de données aberrantes.
6. **Recouper les sources entre elles :** Valider la cohérence croisée.
7. **Un seul système de coordonnées pour les calculs :** Reprojeter les données en **Lambert-93 (EPSG:2154)** pour manipuler des distances métriques fiables.
8. **Tout doit pouvoir se relancer (Idempotence) :** Concevoir un carnet exécutable de bout en bout sans intervention manuelle.


### ⚙️ Bloc 0 : Paramètres Globaux et Configuration
*Objectif : Centraliser toutes les variables de configuration pour garantir l'idempotence du carnet (Règle 8).*

In [94]:
%pip install shapely

StatementMeta(, 868ac63c-5665-4db0-9917-bc77f4b9cce1, 125, Finished, Available, Finished, True)


[notice] A new release of pip is available: 26.1.2 -> 26.3
[notice] To update, run: /nfs4/pyenv-ef740b73-44c4-44e6-83e5-d1adc79b0e88/bin/python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.



In [95]:
# =====================================================================
# BLOC 0 : Paramètres globaux et configuration du pipeline
# =====================================================================
import pandas as pd
import numpy as np
import json
import pyspark.sql.functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType

# 1. Paramètres géographiques (Bounding Box stricte de Lyon)
VILLE_CIBLE = "Lyon"
BBOX_LYON = {"lat_min": 45.68, "lat_max": 45.83, "lon_min": 4.75, "lon_max": 4.93} 

# 2. Système de Coordonnées de Référence (CRS)
CRS_GPS = "EPSG:4326"      # WGS84 (degrés, pour Folium)
CRS_METRIQUE = "EPSG:2154" # Lambert-93 (mètres, pour les calculs de 400m)

# 3. Noms des tables de sortie (Couche Silver)
TABLE_OFFRE = "silver_offre_mobilite"
TABLE_POI = "silver_poi_flux"
TABLE_DEMANDE = "silver_demande_carreaux"
TABLE_CONTEXTE = "silver_contexte_spatial"
TABLE_QUARANTAINE = "silver_quarantaine_rejets"

print(f"🌍 Paramètres chargés pour le territoire : {VILLE_CIBLE}")
print(f"📏 Calculs métriques configurés en : {CRS_METRIQUE}")
print(f"🚨 Table de quarantaine prête : {TABLE_QUARANTAINE}")

StatementMeta(, 868ac63c-5665-4db0-9917-bc77f4b9cce1, 127, Finished, Available, Finished, False)

🌍 Paramètres chargés pour le territoire : Lyon
📏 Calculs métriques configurés en : EPSG:2154
🚨 Table de quarantaine prête : silver_quarantaine_rejets


### 📋 Bloc 1 : Contrats de Schéma (Les Tables Cibles)
*Objectif : Fixer noir sur blanc la structure standardisée de nos tables finales (Règle 1).*

In [96]:
# =====================================================================
# BLOC 1 : Contrats de Schéma (Les moules des tables cibles)
# =====================================================================

# Contrat 1 : L'Offre de Mobilité (Équipements physiques)
SCHEMA_OFFRE_MOBILITE = {
    "id": "string",           
    "source": "string",       # ex: 'velov', 'citiz', 'tcl_arrets'
    "type_service": "string", # ex: 'velo_partage', 'transport_commun', 'recharge_ve'
    "nom": "string",          
    "lat": "double",          
    "lon": "double",          
    "capacite": "integer"     
}

# Contrat 2 : Les POI (Points d'Intérêt Générateurs de Flux)
SCHEMA_POI_FLUX = {
    "id": "string",           
    "source": "string",       # ex: 'hopitaux', 'centres_commerciaux', 'sirene'
    "categorie": "string",    # ex: 'sante', 'commerce', 'emploi'
    "nom": "string",          
    "lat": "double",          
    "lon": "double"           
}

# Contrat 3 : La Demande Démographique (Carroyage) - MISE A JOUR PROJECTIONS
SCHEMA_DEMANDE_CARREAUX = {
    "idcar_200m": "string",         
    "population_actuelle": "float", # 'ind' (2021)
    "menages_actuels": "float",     # 'men'
    "menages_sans_voiture": "float",# Calculé dans le Notebook 1
    "pop_2030": "float",            # Projection Omphale
    "pop_2040": "float",            # Projection Omphale
    "pop_2050": "float"             # Projection Omphale
}

print("📋 Contrats de schéma chargés :")
print(f"   -> {TABLE_OFFRE} ({len(SCHEMA_OFFRE_MOBILITE)} colonnes)")
print(f"   -> {TABLE_POI} ({len(SCHEMA_POI_FLUX)} colonnes)")
print(f"   -> {TABLE_DEMANDE} ({len(SCHEMA_DEMANDE_CARREAUX)} colonnes, avec 3 horizons de projection)")

StatementMeta(, 868ac63c-5665-4db0-9917-bc77f4b9cce1, 128, Finished, Available, Finished, False)

📋 Contrats de schéma chargés :
   -> silver_offre_mobilite (7 colonnes)
   -> silver_poi_flux (6 colonnes)
   -> silver_demande_carreaux (7 colonnes, avec 3 horizons de projection)


### 🛠️ Bloc 2 : Fonctions Socles (Extraction et Quarantaine)
*Objectif : Créer les outils pour extraire les coordonnées de nos géométries JSON (Bronze) et gérer les rejets (Règle 4).*

In [97]:
# =====================================================================
# BLOC 2 : Fonctions Socles (Extraction et Quarantaine)
# =====================================================================

# ---- MACHINE 1 : Extraire des coordonnées ----
# 'def' = Je crée une machine.
# 'extraire_lat_lon' = Le nom de la machine.
# '()' = La zone d'entrée (la prise).
# 'val_geo' = Le nom de l'étiquette qu'on donne à ce qu'on branche dedans.
# (Si on mettait '()' sans rien, la machine refuserait de recevoir quoi que ce soit de l'extérieur).
# ':' = Début des instructions à l'intérieur de la machine.
def extraire_lat_lon(val_geo):
    """
    Cette fonction prend la géométrie brute du Bronze (texte JSON ou dictionnaire)
    et essaie d'en extraire la Latitude et la Longitude.
    """
    # Si la valeur est vide on ne renvoie rien pour eviter un plantage 
    # On utilise 'pd.isna()' et pas 'isnull()' car 'val_geo' est 
    # une valeur isolée (un texte/dict), et non un tableau entier.
    # ('is null' est réservé au SQL, pas au code Python).
    if pd.isna(val_geo):
        # Si la case d'entrée est vide (NaN ou None)...
        # ...on renvoie deux "rien" (None, None) car la machine 
        # doit toujours rendre DEUX valeurs (Latitude et Longitude)
        return None, None

    try:
        # On met un 'try' (essai) au cas où le texte serait corrompu ou illisible.
        
        # CAS 1 : Si la donnée d'entrée est du texte pur ('str')...
        if isinstance(val_geo, str):
            # On utilise '.replace' pour transformer les guillemets simples (') 
            # en guillemets doubles ("), car le traducteur JSON en a absolument besoin.
            # Puis 'json.loads' transforme ce texte nettoyé en un vrai dictionnaire 
            # le plus important c'est vraiment json.loads qui trasforme ce texte en dictionnaire.
            # L'ordi voit '{"type": "Point", "coordinates": [2.35, 48.85]}' comme du texte et c'est impossible à utiliser
            # Python qu'on range dans notre boîte 'geom'.
            import ast
            try:
                geom = ast.literal_eval(val_geo) # ast est plus tolérant
            except:
                geom = json.loads(val_geo.replace("'", '"'))
        
         # CAS 2 : Si la donnée est déjà un dictionnaire Python bien structuré
        elif isinstance(val_geo, dict):
            # Pas besoin de traducteur 'json.loads' ici : 
            # la donnée est déjà propre et directement manipulable.
            # On prend simplement le contenu de 'val_geo' et on le verse 
            # directement dans notre boîte de travail 'geom'.
            geom = val_geo

            # CAS 3 : Si c'est autre chose (chiffre bizarre, etc.), on abandonne
        else:
            return None, None

        # Maintenant qu'on a le dictionnaire, on regarde son type :
        # Maintenant qu'on a le dictionnaire, on regarde son type pour savoir 
        # comment aller chercher les coordonnées à l'intérieur.

        # # On vérifie si la clé normalisée 'type' vaut 'Point' pour savoir 
        # si on traite un point unique, car la structure des coordonnées en dépend.
        # Parce que dans les données géographiques (au format standard GeoJSON), l'objet contient obligatoirement une étiquette nommée "type" qui indique quelle forme géométrique il est.
        # Un point, Multipoint, un polygon
        g_type = geom.get('type')

        # ⚠️ CORRECTION ICI : c'est 'Point' (avec un P majuscule) dans les données !
        if g_type == 'Point':
            # Attention : en cartographie, les coordonnées sont TOUJOURS [Longitude, Latitude] !
            # Donc l'index [1] est la Latitude, et l'index [0] est la Longitude.
            # CORRECTION : Ajout des guillemets autour de 'coordinates' (qui manquaient)
            # CORRECTION BIS : On force le float() pour gérer les array() de TCL
            lon = float(geom['coordinates'][0])
            lat = float(geom['coordinates'][1])
            return lat, lon
        
        # Si c'est un MultiPoint (ex: un Parc Relais)
        elif g_type == 'MultiPoint':
            lon = float(geom['coordinates'][0][0])
            lat = float(geom['coordinates'][0][1])
            return lat, lon

    except Exception as e:
        # CORRECTION DE L'ERREUR SYNTAXIQUE : En Python, un bloc 'try' doit 
        # TOUJOURS être suivi d'un bloc 'sauf que' (except). S'il y a un plantage 
        # dans le try (JSON illisible, clé introuvable), on attrape l'erreur ici 
        # au lieu de tout casser, et on renvoie proprement des valeurs vides.
        return None, None


def envoyer_en_quarantaine(df_rejets, source_origine, motif):
    # Cette fonction accepte exactement 3 entrées distinctes en paramètres.
    # on donne les variables d'entrée que la fonction ne peut pas deviner elle-même
    # Note : le tableau 'df_rejets' n'est pas créé ici ; ce paramètre agit 
    # simplement comme une boîte vide en attente du vrai tableau de données sales.
    # Les deux autres entrées ('source_origine' et 'motif') servent à documenter 
    # précisément d'où vient l'erreur et pourquoi le flux a été rejeté.
    """
    Cette fonction prend les lignes "sales" (rejetées) et les sauvegarde 
    dans une table Delta pour garder une trace (Règle d'or N°4).
    """

    # On vérifie si le tableau des rejets est vide (.empty). S'il l'est, 
    # cela signifie qu'aucune erreur n'a été détectée : il n'y a donc ni données 
    # sales, ni origine, ni motif à enregistrer. L'instruction 'return' arrête net 
    # la fonction pour éviter des calculs et des écritures inutiles en base Delta.
    # Si la liste des rejets est vide, on ne fait rien et on sort de la fonction

    if df_rejets.empty:
        return

    # On fait une copie du tableau pour ne pas modifier l'original par accident
    # On duplique le DataFrame avec '.copy()' pour créer un clone indépendant. 
    # En Python, cela évite de modifier par accident le tableau d'origine 
    # en dehors de la fonction lorsque nous allons lui ajouter des colonnes 
    # de traçabilité juste après.
    df_rejets = df_rejets.copy()

    # On ajoute 3 colonnes pour expliquer l'erreur :
    df_rejets['source_origine'] = source_origine # D'où vient la donnée ? (ex: 'velov')
    df_rejets['motif_rejet'] = motif             # Pourquoi on l'a jetée ? (ex: 'Hors BBox')
    df_rejets['date_traitement'] = pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S") # Quand ?

    # On convertit le tableau Pandas en tableau Spark. 
    # On utilise .astype(str) pour éviter que Spark plante sur des types de données complexes.
    # On convertit le tableau Pandas en DataFrame Spark pour pouvoir l'écrire 
    # dans une table Delta. On utilise '.astype(str)' comme une assurance tous risques : 
    # cela force toutes les colonnes à se transformer en texte brut avant la conversion, 
    # évitant ainsi tout plantage de Spark lié à des types de données complexes ou hétérogènes.
    spark_df = spark.createDataFrame(df_rejets.astype(str))

    # On écrit dans la table Delta de quarantaine
    (spark_df.write
        .format("delta")
        .mode("append") # 'append' veut dire qu'on ajoute à la suite, on n'écrase pas les anciens rejets
        .option("mergeSchema", "true") # Permet d'ajouter de nouvelles colonnes si les sources sont différentes
        .saveAsTable(TABLE_QUARANTAINE)
    )
    # On affiche un message pour prévenir l'utilisateur
    print(f"   🚨 {len(df_rejets)} lignes rejetées envoyées en quarantaine ({motif}).")

print("🛡️ Les deux machines (Fonctions) sont prêtes et chargées en mémoire.")

StatementMeta(, 868ac63c-5665-4db0-9917-bc77f4b9cce1, 129, Finished, Available, Finished, False)

🛡️ Les deux machines (Fonctions) sont prêtes et chargées en mémoire.


### 🔍 Étape 1 : Chargement et Profilage des sources de la Famille A (Audit initial)
*Objectif : Avant de nettoyer, on audite. On charge les 5 tables Bronze, on vérifie leur volumétrie et on cherche les valeurs manquantes pour comprendre à quoi ressemble la donnée brute.*

In [98]:
spark.sql("SHOW TABLES").show(20, truncate=False)

StatementMeta(, 868ac63c-5665-4db0-9917-bc77f4b9cce1, 130, Finished, Available, Finished, False)

+-----------------------------------------+--------------------------------+-----------+
|namespace                                |tableName                       |isTemporary|
+-----------------------------------------+--------------------------------+-----------+
|`Cityz Mobilitee_lyon`.Lyon_lakehouse.dbo|bronze_citiz_stations           |false      |
|`Cityz Mobilitee_lyon`.Lyon_lakehouse.dbo|bronze_dott_stations            |false      |
|`Cityz Mobilitee_lyon`.Lyon_lakehouse.dbo|bronze_filosofi_enrichi_voitures|false      |
|`Cityz Mobilitee_lyon`.Lyon_lakehouse.dbo|bronze_filosofi_revenus         |false      |
|`Cityz Mobilitee_lyon`.Lyon_lakehouse.dbo|bronze_irve_bornes              |false      |
|`Cityz Mobilitee_lyon`.Lyon_lakehouse.dbo|bronze_leogo_vehicules          |false      |
|`Cityz Mobilitee_lyon`.Lyon_lakehouse.dbo|bronze_menages_voitures         |false      |
|`Cityz Mobilitee_lyon`.Lyon_lakehouse.dbo|bronze_parkings_operateurs      |false      |
|`Cityz Mobilitee_lyo

In [99]:
# On vide le cache et on rafraîchit la table pour forcer Spark à lire le nouveau schéma
spark.catalog.clearCache()
spark.catalog.refreshTable("bronze_leogo_vehicules")

StatementMeta(, 868ac63c-5665-4db0-9917-bc77f4b9cce1, 131, Finished, Available, Finished, False)

In [100]:
# =====================================================================
# FAMILLE A : Chargement et Profilage (EDA) des 5 sources de mobilité
# =====================================================================
import pandas as pd

print("🔄 Étape 1.1 : Chargement des 5 sources de mobilité depuis le Lakehouse...")

# =====================================================================
# 🛠️ NOTE TECHNIQUE SUR LE BUG "_source_ttl" (À CONSERVER POUR LA DOC) :
# Lors du premier essai, Spark a planté avec l'erreur "INTERNAL_ERROR_ATTRIBUTE_NOT_FOUND".
# Explication : Dans le Notebook 1, la colonne de traçabilité "_source_ttl" a été 
# remplie avec "None" (vide). Le format Parquet n'a donc pas écrit cette colonne 
# physiquement sur le disque. Cependant, le catalogue Delta Lake en a gardé la trace.
# Résultat : Quand Spark a voulu lire la table, il a cherché une colonne fantôme.
# Solution : La fonction ci-dessous sécurise la lecture en supprimant cette colonne 
# du DataFrame Spark (si elle existe) avant la conversion en Pandas.
# =====================================================================
def charger_table_propre(nom_table):
    df_spark = spark.read.table(nom_table)
    if "_source_ttl" in df_spark.columns:
        df_spark = df_spark.drop("_source_ttl")
    return df_spark.toPandas()

# On lit la table distante depuis le Lakehouse grâce au moteur Big Data (Spark) 
# et on la convertit immédiatement en tableau Pandas local (.toPandas()). 
# Cela permet d'utiliser des outils simples pour auditer et analyser facilement les données.
df_citiz = charger_table_propre("bronze_citiz_stations")
df_velo  = charger_table_propre("bronze_velo_stations")
df_dott  = charger_table_propre("bronze_dott_stations")
df_leogo = charger_table_propre("bronze_leogo_vehicules")
df_tcl   = charger_table_propre("bronze_tcl_arrets")

# Creation du dictionnaire Famille Offre Mobilite 
# On regroupe tous les DataFrames dans un dictionnaire clé-valeur. 
# Cela permet d'utiliser une boucle pour automatiser l'audit de chaque source 
# de mobilité un par un, sans avoir à réécrire le même code cinq fois.
famille_mobilite = {
    "citiz": df_citiz,
    "dott": df_dott,
    "velo": df_velo,
    "leogo": df_leogo, 
    "tcl": df_tcl
}

print("\n🔄 Étape 1.2 : Profilage (Audit) des données...\n")

for nom, df in famille_mobilite.items():
    print(f"--- 📂 Table : {nom.upper()} --- ")

    # On affiche la volumétrie globale (nombre de lignes et de colonnes) de la table. 
    # On utilise ensuite un découpage '[:5]' sur la liste des colonnes pour n'afficher 
    # que les 5 premières, évitant ainsi d'encombrer l'écran si la table est très large.
    print(f"   👉 Volumétrie : {len(df)} lignes, {len(df.columns)} colonnes") 
    print(f"   👉 Colonnes visibles : {list(df.columns)[:5]}...")

    # =================================================================
    # NOUVEAU CONTRÔLE : RECHERCHE DE DOUBLONS (Indispensable pour le ML)
    # =================================================================
    id_col = next((c for c in ['station_id', 'bike_id', 'id'] if c in df.columns), None)
    if id_col:
        nb_doublons = df.duplicated(subset=[id_col]).sum()
        print(f"   👉 Doublons sur la clé '{id_col}' : {nb_doublons} doublon(s).")
    else:
        try:
            nb_doublons = df.duplicated().sum()
            print(f"   👉 Doublons stricts (lignes 100% identiques) : {nb_doublons}")
        except:
            print("   👉 Doublons : Impossible de calculer (colonnes JSON complexes).")

    # =================================================================
    # NOUVEAU CONTRÔLE : FORMAT DU NOM ET CAPACITÉ
    # =================================================================
    if 'name' in df.columns:
        exemple_nom = str(df['name'].iloc[0])[:40] 
        print(f"   👉 Exemple de 'name' : {exemple_nom}...")
        
    capa_cols = [c for c in ['capacity', 'vehicle_capacity'] if c in df.columns]
    if capa_cols:
        print(f"   👉 Colonne de capacité trouvée : '{capa_cols[0]}'")
    else:
        print("   👉 ⚠️ AUCUNE colonne de capacité trouvée.")

    # Analyse des valeurs manquantes (le fameux filtre > 0)
    nuls_par_colonne = df.isnull().sum()
    
    # On filtre le résultat précédent en ne gardant que les colonnes dont 
    # le compteur de vides est strictement supérieur à zéro ('nuls_par_colonne > 0'). 
    # Cela permet d'ignorer les colonnes propres et de ne lister que les anomalies.
    nuls_filtres = nuls_par_colonne[nuls_par_colonne > 0]

    if len(nuls_filtres) == 0:
        print("     ✅ Aucune valeur manquante.")
    else:
        print("     ⚠️ Valeurs manquantes détectées :")

        # On récupère à la fois le nom de la colonne ('col') et son nombre de vides ('count') 
        # en utilisant '.items()' sur notre boîte de sélection 'nuls_filtres'.
        # On calcule ensuite le pourcentage d'anomalies en divisant ce nombre de vides 
        # par la taille totale du tableau ('len(df)') avant de multiplier par 100.
        # Enfin, on affiche un message propre avec un 'print' formaté (f-string) 
        # où le pourcentage est arrondi à un chiffre après la virgule grâce à ':.1f'.
        for col, count in nuls_filtres.items():
            pourcentage = (count / len(df)) * 100
            print(f"       - Colonne '{col}' : {count} vides ({pourcentage:.1f}%)")

    print("") # Ligne vide pour aérer

print("✅ Étape 1 terminée : Les données sont chargées et auditées.")

StatementMeta(, 868ac63c-5665-4db0-9917-bc77f4b9cce1, 132, Finished, Available, Finished, False)

🔄 Étape 1.1 : Chargement des 5 sources de mobilité depuis le Lakehouse...

🔄 Étape 1.2 : Profilage (Audit) des données...

--- 📂 Table : CITIZ --- 
   👉 Volumétrie : 200 lignes, 12 colonnes
   👉 Colonnes visibles : ['station_id', 'name', 'lat', 'lon', 'address']...
   👉 Doublons sur la clé 'station_id' : 0 doublon(s).
   👉 Exemple de 'name' : [{'text': 'Lyon 3ème - Docteur Rebatel',...
   👉 Colonne de capacité trouvée : 'capacity'
     ✅ Aucune valeur manquante.

--- 📂 Table : DOTT --- 
   👉 Volumétrie : 1374 lignes, 11 colonnes
   👉 Colonnes visibles : ['is_virtual_station', 'lat', 'lon', 'name', 'region_id']...
   👉 Doublons sur la clé 'station_id' : 0 doublon(s).
   👉 Exemple de 'name' : Rue Dr horand / en face de parc montel...
   👉 Colonne de capacité trouvée : 'vehicle_capacity'
     ✅ Aucune valeur manquante.

--- 📂 Table : VELO --- 
   👉 Volumétrie : 467 lignes, 11 colonnes
   👉 Colonnes visibles : ['station_id', 'name', 'lat', 'lon', 'address']...
   👉 Doublons sur la clé 'sta

### 🛡️ Étape 2 : Nettoyage Géographique (Bounding Box) et Quarantaine
*Objectif : Extraire les coordonnées GPS, filtrer les points hors de Lyon (avec notre BBox), et séparer les données propres des données sales (Quarantaine) selon la Règle d'Or N°4.*

In [101]:
# =====================================================================
# FAMILLE A : Nettoyage, Quarantaine et Standardisation (Le Moule)
# =====================================================================
import ast  # Bibliothèque pour lire les dictionnaires Python en toute sécurité

print("🧹 Étape 2 : Nettoyage et standardisation de la Famille A...\n")

tables_propres = {}

LAT_MIN, LAT_MAX = BBOX_LYON["lat_min"], BBOX_LYON["lat_max"]
LON_MIN, LON_MAX = BBOX_LYON["lon_min"], BBOX_LYON["lon_max"]

for nom_source, df in famille_mobilite.items():
    print(f"--- Traitement de la source : {nom_source.upper()} ---")
    df_cible = df.copy()
    
    # =================================================================
    # A. EXTRACTION DES COORDONNÉES (Géographie)
    # =================================================================
    if 'geometry' in df_cible.columns:
        # On utilise notre machine "extraire_lat_lon" (qui doit être définie plus haut)
        coords = df_cible['geometry'].apply(extraire_lat_lon)
        df_cible['lat_clean'] = [c[0] if c is not None and len(c) > 0 else None for c in coords]
        df_cible['lon_clean'] = [c[1] if c is not None and len(c) > 1 else None for c in coords]
    elif 'lat' in df_cible.columns:
        df_cible['lat_clean'] = pd.to_numeric(df_cible['lat'], errors='coerce')
        df_cible['lon_clean'] = pd.to_numeric(df_cible['lon'], errors='coerce')
        
    # =================================================================
    # B. FILTRAGE BBOX ET QUARANTAINE
    # =================================================================
    conditions_valides = (
        df_cible['lat_clean'].between(LAT_MIN, LAT_MAX) & 
        df_cible['lon_clean'].between(LON_MIN, LON_MAX)
    )
    
    df_propre = df_cible[conditions_valides].copy()
    df_rejets = df_cible[~conditions_valides].copy()
    
    if not df_rejets.empty:
        envoyer_en_quarantaine(df_rejets, nom_source, "Coordonnées GPS hors BBox Lyon")

    # =================================================================
    # C. STANDARDISATION (Remplissage du Contrat de Schéma)
    # =================================================================
    df_final = pd.DataFrame()
    
    # 1. ID et Source
    id_col = next((c for c in ['station_id', 'bike_id', 'id'] if c in df_propre.columns), 'id')
    df_final['id'] = df_propre[id_col].astype(str)
    df_final['source'] = nom_source
    
    # 2. Coordonnées
    df_final['lat'] = df_propre['lat_clean']
    df_final['lon'] = df_propre['lon_clean']

    # 3. Gestion spécifique du NOM (Texte simple vs JSON/Dict)
    if nom_source == 'citiz':
        # ast.literal_eval lit le dictionnaire Python sans casser sur les apostrophes
        def nom_citiz(x):
            try: return ast.literal_eval(x)[0].get('text', 'Citiz')
            except: return str(x)
        df_final['nom'] = df_propre['name'].apply(nom_citiz)
        df_final['type_service'] = 'autopartage'
        
    elif nom_source == 'tcl':
        # TCL cache son nom dans le dictionnaire 'properties'
        def nom_tcl(x):
            try: return ast.literal_eval(x).get('nom', 'Arrêt TCL')
            except: return 'Arrêt TCL'
        df_final['nom'] = df_propre['properties'].apply(nom_tcl)
        df_final['type_service'] = 'transport_commun'
        
    elif nom_source == 'leogo':
        # Pas de nom, on utilise l'ID du véhicule
        df_final['nom'] = "Leo&Go - " + df_final['id']
        df_final['type_service'] = 'autopartage'
        
    else: # Dott et Vélo'v
        df_final['nom'] = df_propre['name'].astype(str)
        df_final['type_service'] = 'velo_partage' if nom_source == 'velo' else 'trottinette_partage'

    # 4. Gestion spécifique de la CAPACITÉ
    if nom_source == 'dott':
        # Dott a un dictionnaire : {"dott_scooter": 10}
        def capa_dott(x):
            try: return list(ast.literal_eval(x).values())[0]
            except: return 0
        df_final['capacite'] = df_propre['vehicle_capacity'].apply(capa_dott).astype(int)
        
    elif nom_source in ['citiz', 'velo']:
        # Capacité simple
        df_final['capacite'] = pd.to_numeric(df_propre['capacity'], errors='coerce').fillna(0).astype(int)
        
    else:
        # Leo&Go (1 véhicule) et TCL (0 places)
        df_final['capacite'] = 1 if nom_source == 'leogo' else 0

    # On stocke la table moulée
    tables_propres[nom_source] = df_final
    print(f"  ✅ {len(df_final)} lignes moulées et prêtes pour la table finale.\n")

print("🚀 Famille A nettoyée, standardisée et sans bug JSON !")

StatementMeta(, 868ac63c-5665-4db0-9917-bc77f4b9cce1, 133, Finished, Available, Finished, False)

🧹 Étape 2 : Nettoyage et standardisation de la Famille A...

--- Traitement de la source : CITIZ ---
   🚨 10 lignes rejetées envoyées en quarantaine (Coordonnées GPS hors BBox Lyon).
  ✅ 190 lignes moulées et prêtes pour la table finale.

--- Traitement de la source : DOTT ---
   🚨 2 lignes rejetées envoyées en quarantaine (Coordonnées GPS hors BBox Lyon).
  ✅ 1372 lignes moulées et prêtes pour la table finale.

--- Traitement de la source : VELO ---
   🚨 14 lignes rejetées envoyées en quarantaine (Coordonnées GPS hors BBox Lyon).
  ✅ 453 lignes moulées et prêtes pour la table finale.

--- Traitement de la source : LEOGO ---
   🚨 29 lignes rejetées envoyées en quarantaine (Coordonnées GPS hors BBox Lyon).
  ✅ 216 lignes moulées et prêtes pour la table finale.

--- Traitement de la source : TCL ---
   🚨 6311 lignes rejetées envoyées en quarantaine (Coordonnées GPS hors BBox Lyon).
  ✅ 3468 lignes moulées et prêtes pour la table finale.

🚀 Famille A nettoyée, standardisée et sans bug JSO

## 🅿️ Couche Silver : Famille B (Infrastructures & Stationnement)

**Objectif de cette section :**
Poursuivre la structuration de notre Lakehouse en appliquant la même rigueur d'ingénierie (isolation, typage sécurisé, règles métier et politique de quarantaine *Fail Loud*) aux objets géographiques fixes de type **points** (`irve_bornes` et `parkings`).

### 🔍 Étape 1 : Chargement et Profilage (Audit initial)
*Objectif : Importer les tables Bronze de la Famille B et effectuer un premier diagnostic des valeurs manquantes.*

In [102]:
# =====================================================================
# FAMILLE B : Chargement et Profilage (EDA) - Parkings & IRVE
# =====================================================================
import pandas as pd

print("🔄 Étape 1.1 : Chargement des sources de la Famille B...")

# On utilise notre fonction de sécurité définie à l'étape précédente 
# pour éviter le bug du catalogue Spark sur la colonne _source_ttl.
df_irve     = charger_table_propre("bronze_irve_bornes")
df_parkings = charger_table_propre("bronze_parkings_operateurs")

# On regroupe dans un dictionnaire dédié
famille_infrastructure = {
    "irve": df_irve,
    "parkings": df_parkings
}

print("\n🔄 Étape 1.2 : Profilage (Audit) des données...\n")

for nom, df in famille_infrastructure.items():
    print(f"--- 📂 Table : {nom.upper()} --- ")
    
    # 1. VOLUMÉTRIE
    print(f"   👉 Volumétrie : {len(df)} lignes, {len(df.columns)} colonnes")
    
    # 2. DOUBLONS
    # On cherche la colonne d'identifiant unique (souvent 'id' pour les flux OGC)
    id_col = next((c for c in ['id', 'gid'] if c in df.columns), None)
    if id_col:
        nb_doublons = df.duplicated(subset=[id_col]).sum()
        print(f"   👉 Doublons sur la clé '{id_col}' : {nb_doublons} doublon(s).")
    else:
        print("   👉 Doublons : Pas de colonne d'ID évidente trouvée.")

    # 3. ANALYSE DES COLONNES CLÉS (Le fameux JSON 'properties')
    # On regarde à quoi ressemble la première ligne de la colonne 'properties'
    if 'properties' in df.columns:
        exemple_prop = str(df['properties'].iloc[0])
        # On coupe l'affichage pour ne pas polluer la console
        print(f"   👉 Exemple de 'properties' : {exemple_prop[:80]}...")
    else:
        print("   👉 ⚠️ Colonne 'properties' introuvable.")

    # 4. VALEURS MANQUANTES
    nuls_par_colonne = df.isnull().sum()
    nuls_filtres = nuls_par_colonne[nuls_par_colonne > 0]
    
    if len(nuls_filtres) == 0:
        print("   👉 Valeurs manquantes : ✅ Aucune.")
    else:
        print("   👉 Valeurs manquantes :")
        for col, count in nuls_filtres.nlargest(3).items():
            pourcentage = (count / len(df)) * 100
            print(f"       - Colonne '{col}' : {count} vides ({pourcentage:.1f}%)")

    print("") 

print("✅ Étape 1 terminée : Les données de la Famille B sont chargées et auditées.")

StatementMeta(, 868ac63c-5665-4db0-9917-bc77f4b9cce1, 134, Finished, Available, Finished, False)

🔄 Étape 1.1 : Chargement des sources de la Famille B...

🔄 Étape 1.2 : Profilage (Audit) des données...

--- 📂 Table : IRVE --- 
   👉 Volumétrie : 2553 lignes, 10 colonnes
   👉 Doublons sur la clé 'id' : 0 doublon(s).
   👉 Exemple de 'properties' : {'nom_amenageur': 'IKEA', 'siren_amenageur': '', 'contact_amenageur': '', 'nom_o...
   👉 Valeurs manquantes : ✅ Aucune.

--- 📂 Table : PARKINGS --- 
   👉 Volumétrie : 193 lignes, 10 colonnes
   👉 Doublons sur la clé 'id' : 0 doublon(s).
   👉 Exemple de 'properties' : {'gid': 218, 'id': '69282-P-004', 'nom': 'Meyzieu Gare', 'gestionnaire': 'TCL', ...
   👉 Valeurs manquantes : ✅ Aucune.

✅ Étape 1 terminée : Les données de la Famille B sont chargées et auditées.


### 🛡️ Étape 2 : Nettoyage Géographique (Bounding Box) et Quarantaine
*Objectif : Extraire les coordonnées GPS depuis les géométries JSON, filtrer les points hors de Lyon (avec notre BBox), et séparer les données propres des données sales.*

In [103]:
# =====================================================================
# FAMILLE B : Nettoyage, Quarantaine et Standardisation (Le Moule)
# =====================================================================
import ast # Indispensable pour lire le dictionnaire 'properties' en toute sécurité

print("🧹 Étape 2 : Nettoyage et moulage géographique pour la Famille B...\n")

# On prépare un dictionnaire dédié pour stocker les tables propres et formatées
tables_propres_infra = {}

# On récupère les frontières de la BBox
LAT_MIN, LAT_MAX = BBOX_LYON["lat_min"], BBOX_LYON["lat_max"]
LON_MIN, LON_MAX = BBOX_LYON["lon_min"], BBOX_LYON["lon_max"]

# Dictionnaire de configuration pour le moulage des 2 sources
# (On dit à Python où trouver le nom et la capacité dans le dictionnaire 'properties')
config_infra = {
    "irve": {"type": "recharge_ve", "nom_cle": "nom_station", "capa_cle": "nbre_pdc"},
    "parkings": {"type": "parking", "nom_cle": "nom", "capa_cle": "nb_places"}
}

for nom_source, df in famille_infrastructure.items():
    print(f"--- Traitement de la source : {nom_source.upper()} ---")
    
    # Copie de sécurité
    df_cible = df.copy()
    
    # =================================================================
    # A. EXTRACTION DES COORDONNÉES GPS
    # =================================================================
    # Ici, on sait que les coordonnées sont dans la colonne 'geometry' (texte JSON)
    if 'geometry' in df_cible.columns:
        # On applique notre machine "extraire_lat_lon" avec le correctif anti-bug
        coords = df_cible['geometry'].apply(extraire_lat_lon)
        df_cible['lat_clean'] = [c[0] if c is not None and len(c) > 0 else None for c in coords]
        df_cible['lon_clean'] = [c[1] if c is not None and len(c) > 1 else None for c in coords]
    else:
        print(f"  ⚠️ Attention : Impossible de trouver la colonne 'geometry' pour {nom_source}.")
        continue 

    # =================================================================
    # B. FILTRAGE PAR LA BOUNDING BOX ET QUARANTAINE
    # =================================================================
    # Test d'inclusion spatiale
    conditions_valides = (
        df_cible['lat_clean'].between(LAT_MIN, LAT_MAX) & 
        df_cible['lon_clean'].between(LON_MIN, LON_MAX)
    )
    
    # Séparation : Données valides vs Données à rejeter
    df_propre = df_cible[conditions_valides].copy()
    df_rejets = df_cible[~conditions_valides].copy()
    
    # Nettoyage des colonnes temporaires de calcul
    df_propre = df_propre.drop(columns=['lat_clean', 'lon_clean'])
    df_rejets = df_rejets.drop(columns=['lat_clean', 'lon_clean'])
    
    # Envoi à la quarantaine si nécessaire
    if not df_rejets.empty:
        envoyer_en_quarantaine(
            df_rejets=df_rejets, 
            source_origine=nom_source, 
            motif="Coordonnées GPS hors BBox Lyon ou aberrantes"
        )

    # =================================================================
    # C. STANDARDISATION (Le Moulage final)
    # =================================================================
    df_final = pd.DataFrame()
    
    # 1. ID, Source et Type de Service
    df_final['id'] = df_propre['id'].astype(str)
    df_final['source'] = nom_source
    df_final['type_service'] = config_infra[nom_source]["type"]
    
    # 2. Coordonnées (on les recalcule puisqu'on a supprimé lat_clean/lon_clean de df_propre)
    coords_final = df_propre['geometry'].apply(extraire_lat_lon)
    df_final['lat'] = [c[0] if c is not None and len(c) > 0 else None for c in coords_final]
    df_final['lon'] = [c[1] if c is not None and len(c) > 1 else None for c in coords_final]

    # 3. Fonction utilitaire pour extraire une clé du dictionnaire 'properties'
    def extraire_valeur_prop(x, cle, defaut):
        try: 
            # ast.literal_eval transforme le texte en dictionnaire sans planter sur les apostrophes
            return ast.literal_eval(x).get(cle, defaut)
        except: 
            return defaut

    # 4. Extraction du NOM
    nom_cle = config_infra[nom_source]["nom_cle"]
    df_final['nom'] = df_propre['properties'].apply(
        lambda x: extraire_valeur_prop(x, nom_cle, f"{nom_source.capitalize()} sans nom")
    )

    # 5. Extraction de la CAPACITÉ
    capa_cle = config_infra[nom_source]["capa_cle"]
    df_final['capacite'] = df_propre['properties'].apply(
        lambda x: extraire_valeur_prop(x, capa_cle, 0)
    )
    # On s'assure que la capacité est bien un entier
    df_final['capacite'] = pd.to_numeric(df_final['capacite'], errors='coerce').fillna(0).astype(int)

    # On stocke la table moulée
    tables_propres_infra[nom_source] = df_final
    print(f"  ✅ {len(df_final)} lignes moulées et prêtes pour la table finale.\n")

print("🚀 Famille B nettoyée, standardisée et moulée dans le schéma cible !")

StatementMeta(, 868ac63c-5665-4db0-9917-bc77f4b9cce1, 135, Finished, Available, Finished, False)

🧹 Étape 2 : Nettoyage et moulage géographique pour la Famille B...

--- Traitement de la source : IRVE ---
   🚨 545 lignes rejetées envoyées en quarantaine (Coordonnées GPS hors BBox Lyon ou aberrantes).
  ✅ 2008 lignes moulées et prêtes pour la table finale.

--- Traitement de la source : PARKINGS ---
   🚨 22 lignes rejetées envoyées en quarantaine (Coordonnées GPS hors BBox Lyon ou aberrantes).
  ✅ 171 lignes moulées et prêtes pour la table finale.

🚀 Famille B nettoyée, standardisée et moulée dans le schéma cible !


## 🏢 Couche Silver : Famille E (Points d'Intérêt - POI)

**Objectif de cette section :**
Nettoyer les générateurs de flux quotidiens (hôpitaux, centres commerciaux, entreprises SIRENE) pour alimenter la future table `silver_poi_flux`. Ces données sont cruciales pour le Machine Learning, car elles expliquent pourquoi les gens se déplacent.

### 🔍 Étape 1 : Chargement et Profilage (Audit initial)
*Objectif : Charger les 3 tables Bronze et comprendre leur structure (surtout pour SIRENE qui n'a pas de coordonnées GPS directes).*

In [104]:
# =====================================================================
# FAMILLE C : Chargement et Profilage (EDA) - Population
# =====================================================================
import pyspark.sql.functions as F

print("🔄 Étape 1.1 : Chargement de la table Socio-démographique depuis le Lakehouse...")

# On charge la table en Spark (pas en Pandas !) car elle contient 2,3 millions de lignes.
df_pop_spark = spark.read.table("bronze_filosofi_enrichi_voitures")

print("\n🔄 Étape 1.2 : Profilage (Audit) des données...\n")

print(f"--- 📂 Table : FILOSOFI ENRICHI --- ")

# 1. VOLUMÉTRIE
nb_lignes = df_pop_spark.count()
nb_colonnes = len(df_pop_spark.columns)
print(f"   👉 Volumétrie : {nb_lignes} lignes, {nb_colonnes} colonnes")

# 2. COLONNES DISPONIBLES
# On affiche toutes les colonnes pour vérifier que nos calculs du Notebook 1 sont bien là
print(f"   👉 Colonnes disponibles :")
print(f"      {df_pop_spark.columns}")

# 3. RECHERCHE DE DOUBLONS (Sur la clé spatiale 'idcar_200m')
# On vérifie si un carreau est présent plusieurs fois
nb_doublons = nb_lignes - df_pop_spark.select("idcar_200m").distinct().count()
print(f"   👉 Doublons sur la clé 'idcar_200m' : {nb_doublons} doublon(s).")

# 4. VALEURS MANQUANTES (Sur les colonnes clés)
# On compte les vides sur les colonnes importantes
print("   👉 Valeurs manquantes sur les colonnes clés :")
for col in ["idcar_200m", "ind", "est_menages_sans_voiture", "ind_2050"]:
    if col in df_pop_spark.columns:
        nb_nuls = df_pop_spark.filter(F.col(col).isNull()).count()
        print(f"      - Colonne '{col}' : {nb_nuls} vides")

print("\n✅ Étape 1 terminée : Les données démographiques sont chargées et auditées.")

StatementMeta(, 868ac63c-5665-4db0-9917-bc77f4b9cce1, 136, Finished, Available, Finished, False)

🔄 Étape 1.1 : Chargement de la table Socio-démographique depuis le Lakehouse...

🔄 Étape 1.2 : Profilage (Audit) des données...

--- 📂 Table : FILOSOFI ENRICHI --- 
   👉 Volumétrie : 2298582 lignes, 20 colonnes
   👉 Colonnes disponibles :
      ['idcar_200m', 'ind', 'men', 'men_pauv', 'men_prop', 'men_surf', 'men_coll', 'men_mais', 'log_av45', '_source_url', '_source_last_updated', '_source_ttl', '_source_version', '_ingestion_ts', 'ind_2030', 'ind_2040', 'ind_2050', 'taux_sans_voiture_calibre', 'est_menages_sans_voiture', 'est_menages_avec_voiture']
   👉 Doublons sur la clé 'idcar_200m' : 0 doublon(s).
   👉 Valeurs manquantes sur les colonnes clés :
      - Colonne 'idcar_200m' : 0 vides
      - Colonne 'ind' : 0 vides
      - Colonne 'est_menages_sans_voiture' : 0 vides
      - Colonne 'ind_2050' : 0 vides

✅ Étape 1 terminée : Les données démographiques sont chargées et auditées.


### 🛡️ Étape 2 : Nettoyage Géographique (Bounding Box) et Quarantaine
*Objectif : Extraire les coordonnées GPS (que ce soit via JSON ou colonnes latitude/longitude), filtrer les points hors de Lyon, et envoyer les anomalies en quarantaine.*

In [105]:
# =====================================================================
# FAMILLE C : Nettoyage et Moulage (Population)
# =====================================================================
import pyspark.sql.functions as F

print("🧹 Étape 2 : Nettoyage et moulage de la Famille C (Spark Natif)...\n")

# 1. On charge la table brute
df_pop_brut = spark.read.table("bronze_filosofi_enrichi_voitures")

# =====================================================================
# A. DÉFINITION DES CONDITIONS MÉTIER (Avec Cast explicite)
# =====================================================================
# On force le passage en "double" (chiffre à virgule) car la table a été sauvée en texte
condition_valide = (
    (F.col("ind").cast("double") >= 0) & 
    (F.col("men").cast("double") >= 0) & 
    (F.col("est_menages_sans_voiture").cast("double") >= 0)
)

# Séparation des données
df_valide_spark = df_pop_brut.filter(condition_valide)
df_rejets_spark = df_pop_brut.filter(~condition_valide)

# =====================================================================
# B. ENVOI À LA QUARANTAINE (Sécurisé)
# =====================================================================
# On vérifie d'abord COMBIEN il y a de rejets, sans les charger en mémoire
nb_rejets = df_rejets_spark.count()

if nb_rejets > 0:
    print(f"  🚨 {nb_rejets} lignes de population invalides détectées.")
    # Si on en a (ce qui est très rare), on les charge en mémoire
    df_rejets_pd = df_rejets_spark.limit(10000).toPandas() # limit() par sécurité
    envoyer_en_quarantaine(
        df_rejets=df_rejets_pd, 
        source_origine="filosofi_population", 
        motif="Valeurs de population ou de ménages négatives"
    )
else:
    print("  ✨ Aucune ligne invalide détectée pour la population (0 rejet).")

# =====================================================================
# C. STANDARDISATION (Le Moulage final)
# =====================================================================
# On sélectionne et renomme les colonnes, et on s'assure qu'elles sont bien numériques
df_silver_spark = df_valide_spark.select(
    F.col("idcar_200m").alias("carreau_id"),
    F.col("ind").cast("double").alias("population_actuelle"),
    F.col("men").cast("double").alias("menages_actuels"),
    F.col("est_menages_sans_voiture").cast("double").alias("menages_sans_voiture"),
    F.col("ind_2030").cast("double").alias("pop_2030"),
    F.col("ind_2040").cast("double").alias("pop_2040"),
    F.col("ind_2050").cast("double").alias("pop_2050")
)

# =====================================================================
# D. ÉCRITURE DE LA TABLE SILVER
# =====================================================================
print("⏳ Enregistrement de la table Delta 'silver_demande_carreaux'...")

(
    df_silver_spark.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver_demande_carreaux")
)

print(f"🎉 Succès ! Table Silver enregistrée avec {df_silver_spark.count()} lignes.")

StatementMeta(, 868ac63c-5665-4db0-9917-bc77f4b9cce1, 137, Finished, Available, Finished, False)

🧹 Étape 2 : Nettoyage et moulage de la Famille C (Spark Natif)...

  ✨ Aucune ligne invalide détectée pour la population (0 rejet).
⏳ Enregistrement de la table Delta 'silver_demande_carreaux'...
🎉 Succès ! Table Silver enregistrée avec 2298582 lignes.


## 🛣️ Couche Silver : Famille D (Voirie & Aménagements)

**Objectif de cette section :**
Analyser le réseau de voirie (chaussées et trottoirs) pour préparer les futurs calculs d'accessibilité piétonne. La voirie est un cas particulier : ce sont des lignes (LineString) et non des points.

### 🔍 Étape 1 : Chargement et Profilage (Audit initial)
*Objectif : Charger la table de la voirie, vérifier la volumétrie, la présence de doublons et la structure de la géométrie.*

In [106]:
# =====================================================================
# FAMILLE D : Chargement et Profilage (EDA) - Voirie
# =====================================================================
import pandas as pd
import pyspark.sql.functions as F

print("🔄 Étape 1.1 : Chargement de la table Voirie depuis le Lakehouse...")

# On utilise notre fonction de sécurité anti-bug (_source_ttl)
df_voirie = charger_table_propre("bronze_pvo_voirie")

# On met la table dans un dictionnaire (même s'il n'y a qu'une source) 
# pour garder la même logique que les autres familles.
famille_voirie = {
    "voirie": df_voirie
}

print("\n🔄 Étape 1.2 : Profilage (Audit) des données...\n")

for nom, df in famille_voirie.items():
    print(f"--- 📂 Table : {nom.upper()} --- ")
    
    # 1. VOLUMÉTRIE
    print(f"   👉 Volumétrie : {len(df)} lignes, {len(df.columns)} colonnes")
    
    # 2. DOUBLONS
    # On cherche un identifiant unique de tronçon (gid ou codetroncon)
    id_col = next((c for c in ['gid', 'codetroncon', 'id'] if c in df.columns), None)
    if id_col:
        nb_doublons = df.duplicated(subset=[id_col]).sum()
        print(f"   👉 Doublons sur la clé '{id_col}' : {nb_doublons} doublon(s).")
    else:
        print("   👉 Doublons : Pas de colonne d'ID évidente trouvée.")

    # 3. RECHERCHE DES COORDONNÉES (Le point critique pour la voirie)
    # On vérifie si on a bien une colonne geometry
    if 'geometry' in df.columns:
        # On regarde à quoi ressemble la première ligne (c'est du JSON)
        exemple_geom = str(df['geometry'].iloc[0])[:100]
        print(f"   👉 Exemple de 'geometry' : {exemple_geom}...")
    else:
        print("   👉 ⚠️ Colonne 'geometry' introuvable.")

    # 4. APERÇU DES COLONNES CLÉS
    print(f"   👉 Colonnes visibles : {list(df.columns)[:5]}...")

    # 5. VALEURS MANQUANTES
    nuls_par_colonne = df.isnull().sum()
    nuls_filtres = nuls_par_colonne[nuls_par_colonne > 0]
    
    if len(nuls_filtres) == 0:
        print("   👉 Valeurs manquantes : ✅ Aucune.")
    else:
        print("   👉 Valeurs manquantes :")
        # On affiche les 3 colonnes les plus vides
        for col, count in nuls_filtres.nlargest(3).items():
            pourcentage = (count / len(df)) * 100
            print(f"       - Colonne '{col}' : {count} vides ({pourcentage:.1f}%)")

    print("") 

print("✅ Étape 1 terminée : La table Voirie est chargée et auditée.")

StatementMeta(, 868ac63c-5665-4db0-9917-bc77f4b9cce1, 138, Finished, Available, Finished, False)

🔄 Étape 1.1 : Chargement de la table Voirie depuis le Lakehouse...

🔄 Étape 1.2 : Profilage (Audit) des données...

--- 📂 Table : VOIRIE --- 
   👉 Volumétrie : 37569 lignes, 57 colonnes
   👉 Doublons sur la clé 'gid' : 0 doublon(s).
   👉 Exemple de 'geometry' : LINESTRING (4.827772 45.699192, 4.829156 45.698944, 4.829255 45.698922, 4.829338 45.698878, 4.829424...
   👉 Colonnes visibles : ['geometry', 'codetroncon', 'codefuv', 'nomvoie1', 'commune1']...
   👉 Valeurs manquantes : ✅ Aucune.

✅ Étape 1 terminée : La table Voirie est chargée et auditée.


### 🛡️ Étape 2 : Nettoyage Géographique (Centroïde) et Quarantaine
*Objectif : Comme la voirie est composée de lignes (LineString), on calcule le centroïde (point central) de chaque tronçon pour pouvoir appliquer notre filtre BBox. On isole aussi les lignes hors de Lyon.*

In [107]:
# =====================================================================
# FAMILLE D : Nettoyage, Centroïde et Quarantaine (Voirie)
# =====================================================================
import shapely.wkt

print("🧹 Étape 2 : Début du nettoyage géographique pour la Voirie...\n")

tables_propres_voirie = {}

LAT_MIN, LAT_MAX = BBOX_LYON["lat_min"], BBOX_LYON["lat_max"]
LON_MIN, LON_MAX = BBOX_LYON["lon_min"], BBOX_LYON["lon_max"]

def extraire_centroide_wkt(geom_val):
    """
    Prend un texte WKT (ex: 'LINESTRING (4.8 45.7, ...)') 
    et renvoie le centroïde (Latitude, Longitude).
    """
    if pd.isna(geom_val):
        return None, None
    try:
        # On transforme le texte WKT en objet géométrique Shapely
        geom = shapely.wkt.loads(geom_val)
        # On calcule le centre de la ligne
        centroid = geom.centroid
        # On retourne (Lat, Lon)
        return centroid.y, centroid.x
    except Exception:
        return None, None

for nom_source, df in famille_voirie.items():
    print(f"--- Traitement de la source : {nom_source.upper()} ---")
    
    df_cible = df.copy()
    
    # =================================================================
    # A. EXTRACTION DU CENTROÏDE
    # =================================================================
    coords = df_cible['geometry'].apply(extraire_centroide_wkt)
    df_cible['lat_clean'] = [c[0] if c is not None and len(c) > 0 else None for c in coords]
    df_cible['lon_clean'] = [c[1] if c is not None and len(c) > 1 else None for c in coords]

    # =================================================================
    # B. FILTRAGE PAR LA BOUNDING BOX ET QUARANTAINE
    # =================================================================
    conditions_valides = (
        df_cible['lat_clean'].between(LAT_MIN, LAT_MAX) & 
        df_cible['lon_clean'].between(LON_MIN, LON_MAX)
    )
    
    df_propre = df_cible[conditions_valides].copy()
    df_rejets = df_cible[~conditions_valides].copy()
    
    # Nettoyage des colonnes temporaires
    df_propre = df_propre.drop(columns=['lat_clean', 'lon_clean'])
    df_rejets = df_rejets.drop(columns=['lat_clean', 'lon_clean'])
    
    if not df_rejets.empty:
        envoyer_en_quarantaine(df_rejets, nom_source, "Centroïde hors BBox Lyon")

    # =================================================================
    # C. STANDARDISATION (Sélection des colonnes utiles)
    # =================================================================
    colonnes_utiles = [
        'gid', 'codetroncon', 'nomvoie1', 'commune1', 
        'largeurchaussee', 'largeurtrottoirdroit', 'largeurtrottoirgauche',
        'limitationvitesse', 'geometry'
    ]
    
    cols_presentes = [c for c in colonnes_utiles if c in df_propre.columns]
    df_final = df_propre[cols_presentes].copy()
    
    tables_propres_voirie[nom_source] = df_final
    print(f"  ✅ {len(df_final)} lignes valides conservées pour la Voirie.\n")

print("🚀 Famille D nettoyée et prête !")

StatementMeta(, 868ac63c-5665-4db0-9917-bc77f4b9cce1, 139, Finished, Available, Finished, False)

🧹 Étape 2 : Début du nettoyage géographique pour la Voirie...

--- Traitement de la source : VOIRIE ---
   🚨 15945 lignes rejetées envoyées en quarantaine (Centroïde hors BBox Lyon).
  ✅ 21624 lignes valides conservées pour la Voirie.

🚀 Famille D nettoyée et prête !


## 🏢 Couche Silver : Famille E (Points d'Intérêt - POI)

**Objectif de cette section :**
Analyser les générateurs de flux quotidiens (hôpitaux, centres commerciaux, entreprises SIRENE) pour préparer la future table `silver_poi_flux`. Ces données sont cruciales pour le Machine Learning, car elles expliquent pourquoi les gens se déplacent.

### 🔍 Étape 1 : Chargement et Profilage (Audit initial)
*Objectif : Charger les 3 tables Bronze et comprendre leur structure. On vérifie notamment comment SIRENE stocke ses coordonnées GPS, puisqu'elle ne possède pas de colonne `geometry` standard.*

In [109]:
# =====================================================================
# FAMILLE E : Chargement et Profilage (EDA) - POI (Version Corrigée)
# =====================================================================
import pandas as pd

print("🔄 Étape 1.1 : Chargement des sources de la Famille E...")

# On utilise la fonction de sécurité anti-bug _source_ttl
df_commerces = charger_table_propre("bronze_poi_commerces")
df_sante     = charger_table_propre("bronze_poi_sante")
df_sirene    = charger_table_propre("bronze_poi_sirene")

# On regroupe dans un dictionnaire dédié
famille_poi = {
    "commerces": df_commerces,
    "sante": df_sante,
    "sirene": df_sirene
}

print("\n🔄 Étape 1.2 : Profilage (Audit) des données...\n")

for nom, df in famille_poi.items():
    print(f"--- 📂 Table : {nom.upper()} --- ")
    
    # 1. VOLUMÉTRIE
    print(f"   👉 Volumétrie : {len(df)} lignes, {len(df.columns)} colonnes")
    
    # 2. RECHERCHE DE DOUBLONS
    id_col = next((c for c in ['siret', 'siren', 'id', 'gid'] if c in df.columns), None)
    if id_col:
        nb_doublons = df.duplicated(subset=[id_col]).sum()
        print(f"   👉 Doublons sur la clé '{id_col}' : {nb_doublons} doublon(s).")
    else:
        print("   👉 Doublons : Pas de colonne d'ID évidente trouvée.")

    # 3. RECHERCHE DES COORDONNÉES GPS (Corrigé avec les vrais noms)
    col_geom = next((c for c in ['geometry_raw', 'the_geom', 'geometry'] if c in df.columns), None)
    
    if col_geom:
        # On affiche un exemple pour voir le format (JSON ou WKT)
        exemple_geom = str(df[col_geom].iloc[0])[:80]
        print(f"   👉 Coordonnées trouvées dans '{col_geom}' (ex: {exemple_geom}...)")
    else:
        print("   👉 Coordonnées : ⚠️ AUCUNE colonne de géométrie trouvée.")

    # 4. APERÇU DES COLONNES CLÉS (Pour voir où se cache le nom)
    print(f"   👉 Colonnes visibles : {list(df.columns)[:5]}...")

    # 5. VALEURS MANQUANTES
    nuls_par_colonne = df.isnull().sum()
    nuls_filtres = nuls_par_colonne[nuls_par_colonne > 0]
    
    if len(nuls_filtres) == 0:
        print("   👉 Valeurs manquantes : ✅ Aucune.")
    else:
        print("   👉 Valeurs manquantes :")
        for col, count in nuls_filtres.nlargest(3).items():
            pourcentage = (count / len(df)) * 100
            print(f"       - Colonne '{col}' : {count} vides ({pourcentage:.1f}%)")

    print("") 

print("✅ Étape 1 terminée : Les données POI sont chargées et auditées correctement.")

StatementMeta(, 868ac63c-5665-4db0-9917-bc77f4b9cce1, 141, Finished, Available, Finished, False)

🔄 Étape 1.1 : Chargement des sources de la Famille E...

🔄 Étape 1.2 : Profilage (Audit) des données...

--- 📂 Table : COMMERCES --- 
   👉 Volumétrie : 29 lignes, 14 colonnes
   👉 Doublons sur la clé 'gid' : 0 doublon(s).
   👉 Coordonnées trouvées dans 'geometry_raw' (ex: {'type': 'Polygon', 'coordinates': [[[4.857092, 45.802866], [4.857125, 45.802886...)
   👉 Colonnes visibles : ['nom', 'commune', 'insee', 'adresse', 'address']...
   👉 Valeurs manquantes : ✅ Aucune.

--- 📂 Table : SANTE --- 
   👉 Volumétrie : 54 lignes, 11 colonnes
   👉 Doublons sur la clé 'siret' : 3 doublon(s).
   👉 Coordonnées trouvées dans 'geometry_raw' (ex: {'type': 'MultiPolygon', 'coordinates': [[[[4.8971764, 45.74807976], [4.90017214...)
   👉 Colonnes visibles : ['nom', 'idexterne', 'siret', 'datecreation', 'datemaj']...
   👉 Valeurs manquantes : ✅ Aucune.

--- 📂 Table : SIRENE --- 
   👉 Volumétrie : 324971 lignes, 20 colonnes
   👉 Doublons sur la clé 'siret' : 0 doublon(s).
   👉 Coordonnées trouvées dans 'th

### 🛡️ Étape 2 : Nettoyage, Déduplication et Standardisation (POI)
*Objectif : Nettoyer les données de la Famille E (Commerces, Santé, SIRENE). On va supprimer les 3 doublons de la table Santé, extraire le centroïde des polygones (Commerces/Santé) et les coordonnées des points WKT (SIRENE), puis filtrer avec la Bounding Box et envoyer les rejets en quarantaine.*

In [110]:
# =====================================================================
# FAMILLE E : Nettoyage, Déduplication et Standardisation (POI)
# =====================================================================
import pandas as pd
import ast
import shapely.wkt
from shapely.geometry import shape

print("🧹 Étape 2 : Nettoyage et moulage de la Famille E...\n")

# 🛡️ SÉCURITÉ ANTI-REDÉMARRAGE
BBOX_LYON = {"lat_min": 45.68, "lat_max": 45.83, "lon_min": 4.75, "lon_max": 4.93}
LAT_MIN, LAT_MAX = BBOX_LYON["lat_min"], BBOX_LYON["lat_max"]
LON_MIN, LON_MAX = BBOX_LYON["lon_min"], BBOX_LYON["lon_max"]

tables_propres_poi = {}

# Fonction robuste pour extraire les coordonnées (gère JSON Polygons et WKT Points)
def extraire_coord_poi(val):
    if pd.isna(val): return None, None
    try:
        if isinstance(val, str) and val.strip().startswith('{'):
            # Format JSON (Commerces / Santé) : c'est un Polygone
            geom_dict = ast.literal_eval(val)
            poly = shape(geom_dict)
            centroid = poly.centroid
            return centroid.y, centroid.x # lat, lon
        elif isinstance(val, str) and val.upper().startswith('POINT'):
            # Format WKT (SIRENE) : c'est un Point
            coords_str = val.replace('POINT', '').replace('(', '').replace(')', '').strip()
            lon, lat = map(float, coords_str.split())
            return lat, lon
    except Exception:
        return None, None
    return None, None

for nom_source, df in famille_poi.items():
    print(f"--- Traitement de la source : {nom_source.upper()} ---")
    
    df_cible = df.copy()
    
    # =================================================================
    # A. DÉDUPLICATION (En priorité sur la Santé)
    # =================================================================
    id_col = next((c for c in ['siret', 'siren', 'gid', 'id'] if c in df_cible.columns), None)
    if id_col:
        avant = len(df_cible)
        df_cible = df_cible.drop_duplicates(subset=[id_col], keep='first')
        apres = len(df_cible)
        if avant != apres:
            print(f"  🧹 {avant - apres} doublons supprimés sur la clé '{id_col}'.")

    # =================================================================
    # B. EXTRACTION DES COORDONNÉES GPS
    # =================================================================
    col_geom = next((c for c in ['geometry_raw', 'the_geom'] if c in df_cible.columns), None)
    
    if col_geom:
        coords = df_cible[col_geom].apply(extraire_coord_poi)
        df_cible['lat_clean'] = [c[0] if c is not None else None for c in coords]
        df_cible['lon_clean'] = [c[1] if c is not None else None for c in coords]
    else:
        df_cible['lat_clean'] = None
        df_cible['lon_clean'] = None

    # =================================================================
    # C. FILTRAGE BBOX ET QUARANTAINE
    # =================================================================
    conditions_valides = (
        df_cible['lat_clean'].between(LAT_MIN, LAT_MAX) & 
        df_cible['lon_clean'].between(LON_MIN, LON_MAX)
    )
    
    df_propre = df_cible[conditions_valides].copy()
    df_rejets = df_cible[~conditions_valides].copy()
    
    if not df_rejets.empty:
        if 'envoyer_en_quarantaine' not in locals():
            def envoyer_en_quarantaine(df_rejets, source, motif):
                df_rejets['source_origine'] = source; df_rejets['motif_rejet'] = motif
                spark.createDataFrame(df_rejets.astype(str)).write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable("silver_quarantaine_rejets")
        envoyer_en_quarantaine(df_rejets, nom_source, "Coordonnées GPS hors BBox Lyon")

    # =================================================================
    # D. STANDARDISATION (Moulage SCHEMA_POI_FLUX)
    # =================================================================
    df_final = pd.DataFrame()
    
    # 1. ID, Source, Catégorie
    df_final['id'] = df_propre[id_col].astype(str) if id_col else nom_source + "_" + df_propre.index.astype(str)
    df_final['source'] = nom_source
    
    cat_map = {"sante": "sante", "commerces": "commerce", "sirene": "emploi"}
    df_final['categorie'] = cat_map.get(nom_source, "autre")
    
    # 2. Nom
    nom_col = next((c for c in ['nom', 'denomination', 'libelle'] if c in df_propre.columns), None)
    df_final['nom'] = df_propre[nom_col].astype(str) if nom_col else f"{nom_source.capitalize()} sans nom"

    # 3. Coordonnées
    df_final['lat'] = [c[0] if c is not None else None for c in df_propre[col_geom].apply(extraire_coord_poi)] if col_geom else None
    df_final['lon'] = [c[1] if c is not None else None for c in df_propre[col_geom].apply(extraire_coord_poi)] if col_geom else None

    tables_propres_poi[nom_source] = df_final
    print(f"  ✅ {len(df_final)} lignes moulées et prêtes.\n")

print("🚀 Famille E nettoyée et moulée !")

StatementMeta(, 868ac63c-5665-4db0-9917-bc77f4b9cce1, 142, Finished, Available, Finished, False)

🧹 Étape 2 : Nettoyage et moulage de la Famille E...

--- Traitement de la source : COMMERCES ---
   🚨 9 lignes rejetées envoyées en quarantaine (Coordonnées GPS hors BBox Lyon).
  ✅ 20 lignes moulées et prêtes.

--- Traitement de la source : SANTE ---
  🧹 3 doublons supprimés sur la clé 'siret'.
   🚨 6 lignes rejetées envoyées en quarantaine (Coordonnées GPS hors BBox Lyon).
  ✅ 45 lignes moulées et prêtes.

--- Traitement de la source : SIRENE ---
   🚨 56442 lignes rejetées envoyées en quarantaine (Coordonnées GPS hors BBox Lyon).
  ✅ 268529 lignes moulées et prêtes.

🚀 Famille E nettoyée et moulée !


## 🏗️ Étape 3 : L'Assemblage Final - Création de `silver_offre_mobilite`

**Objectif :** Nous avons nettoyé, filtré et moulé nos données famille par famille. Il est temps de les assembler pour créer notre table Silver de l'Offre.

### 🧠 Pourquoi assembler la Famille A et la Famille B ?
Dans notre modèle de géomarketing, la Famille A (Citiz, Vélo'v, TCL...) et la Famille B (Parkings, IRVE) sont toutes deux des **points physiques d'accueil de mobilité**. 
Les fusionner permet à nos futurs algorithmes (Couche Gold) de faire une seule requête spatiale : *"Quels sont tous les points de mobilité autour de ce carreau ?"*

### 🛠️ Comment on les assemble ?
Puisque nous avons appliqué notre **Contrat de Schéma** (les 7 colonnes : `id`, `source`, `type_service`, `nom`, `lat`, `lon`, `capacite`) lors de l'étape de Moulage, l'empilement se fait sans aucune erreur. 
*   La colonne `source` nous dira d'où vient la donnée (ex: 'velov').
*   La colonne `type_service` nous dira de quoi il s'agit (ex: 'velo_partage').


In [111]:
# =====================================================================
# ASSEMBLAGE : Création de la table silver_offre_mobilite
# =====================================================================
import pandas as pd

print("🧱 Étape 3 : Début de l'assemblage des Familles A et B...\n")

# 1. On récupère toutes les tables propres de la Famille A (tables_propres)
# et de la Famille B (tables_propres_infra) dans une grande liste.
liste_des_tables = []
liste_des_tables.extend(tables_propres.values())
liste_des_tables.extend(tables_propres_infra.values())

# 2. L'EMPILEMENT (Concaténation)
# pd.concat permet de mettre les tables les unes sous les autres.
# ignore_index=True réinitialise la numérotation des lignes pour éviter les doublons d'index.
df_offre_final = pd.concat(liste_des_tables, ignore_index=True)

print(f"✅ Empilement terminé : {len(df_offre_final)} lignes au total.")

# 3. VÉRIFICATION DU CONTRAT
# On s'assure que le tableau final possède bien les 7 colonnes exigées.
colonnes_attendues = ['id', 'source', 'type_service', 'nom', 'lat', 'lon', 'capacite']
# On réorganise les colonnes dans le bon ordre
df_offre_final = df_offre_final[colonnes_attendues]

# 4. GESTION DES VALEURS MANQUANTES
# Au cas où une coordonnée aurait échoué, on supprime la ligne.
avant_nettoyage = len(df_offre_final)
df_offre_final = df_offre_final.dropna(subset=['lat', 'lon'])
if avant_nettoyage != len(df_offre_final):
    print(f"🧹 {avant_nettoyage - len(df_offre_final)} lignes sans coordonnées GPS ont été retirées.")

# 5. SAUVEGARDE DANS LE LAKEHOUSE
print("\n⏳ Écriture de la table Delta 'silver_offre_mobilite'...")

# Conversion en Spark DataFrame pour l'écriture distribuée
spark_df_offre = spark.createDataFrame(df_offre_final)

# Écriture en mode overwrite avec mise à jour du schéma
(
    spark_df_offre.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver_offre_mobilite")
)

print(f"🎉 Succès ! Table 'silver_offre_mobilite' enregistrée avec {len(df_offre_final)} lignes !")

StatementMeta(, 868ac63c-5665-4db0-9917-bc77f4b9cce1, 143, Finished, Available, Finished, False)

🧱 Étape 3 : Début de l'assemblage des Familles A et B...

✅ Empilement terminé : 7878 lignes au total.

⏳ Écriture de la table Delta 'silver_offre_mobilite'...
🎉 Succès ! Table 'silver_offre_mobilite' enregistrée avec 7878 lignes !


### 2. Création de `silver_poi_flux`
*Objectif : Assembler les tables de la Famille E (Commerces, Santé, SIRENE) pour créer la table Silver des générateurs de flux. C'est cette table qui sera croisée avec la demande (Filosofi) pour le Machine Learning.*

In [112]:
# =====================================================================
# ASSEMBLAGE : Création de la table silver_poi_flux
# =====================================================================
import pandas as pd

print("🧱 Étape 3 : Début de l'assemblage de la Famille E (POI)...\n")

# 1. On récupère toutes les tables POI propres de la Famille E
liste_poi = list(tables_propres_poi.values())

# 2. L'EMPILEMENT
df_poi_final = pd.concat(liste_poi, ignore_index=True)

print(f"✅ Empilement terminé : {len(df_poi_final)} lignes au total.")

# 3. VÉRIFICATION DU CONTRAT
colonnes_attendues = ['id', 'source', 'categorie', 'nom', 'lat', 'lon']
df_poi_final = df_poi_final[colonnes_attendues]

# 4. GESTION DES VALEURS MANQUANTES (Au cas où)
df_poi_final = df_poi_final.dropna(subset=['lat', 'lon'])

# 5. SAUVEGARDE DANS LE LAKEHOUSE
print("\n⏳ Écriture de la table Delta 'silver_poi_flux'...")

spark_df_poi = spark.createDataFrame(df_poi_final)

(
    spark_df_poi.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver_poi_flux")
)

print(f"🎉 Succès ! Table 'silver_poi_flux' enregistrée avec {len(df_poi_final)} lignes !")

StatementMeta(, 868ac63c-5665-4db0-9917-bc77f4b9cce1, 144, Finished, Available, Finished, False)

🧱 Étape 3 : Début de l'assemblage de la Famille E (POI)...

✅ Empilement terminé : 268594 lignes au total.

⏳ Écriture de la table Delta 'silver_poi_flux'...
🎉 Succès ! Table 'silver_poi_flux' enregistrée avec 268594 lignes !


In [113]:
# =====================================================================
# SAUVEGARDE : Création de la table silver_contexte_spatial (Voirie)
# =====================================================================
import pandas as pd

print("🧱 Étape 4 : Sauvegarde de la Voirie dans la Couche Silver...\n")

# 1. On récupère la table de la voirie nettoyée (mise en mémoire plus tôt)
df_voirie_final = tables_propres_voirie["voirie"]

print(f"✅ Volumétrie : {len(df_voirie_final)} tronçons de voirie.")

# 2. SAUVEGARDE DANS LE LAKEHOUSE
print("\n⏳ Écriture de la table Delta 'silver_contexte_spatial'...")

# On convertit en Spark pour l'écriture
spark_df_voirie = spark.createDataFrame(df_voirie_final.astype(str))

(
    spark_df_voirie.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver_contexte_spatial")
)

print(f"🎉 Succès ! Table 'silver_contexte_spatial' enregistrée avec {len(df_voirie_final)} lignes !")

StatementMeta(, 868ac63c-5665-4db0-9917-bc77f4b9cce1, 145, Finished, Available, Finished, False)

🧱 Étape 4 : Sauvegarde de la Voirie dans la Couche Silver...

✅ Volumétrie : 21624 tronçons de voirie.

⏳ Écriture de la table Delta 'silver_contexte_spatial'...
🎉 Succès ! Table 'silver_contexte_spatial' enregistrée avec 21624 lignes !
